In [1]:
%pip install -e .

Obtaining file:///C:/Users/david/programmier/Master_Programmier
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for thechem_live (pyproject.toml): started
  Building editable for thechem_live (pyproject.toml): finished with status 'done'
  Created wheel for thechem_live: filename=thechem_live-0.1.0-0.editable-py3-none-any.whl size=2748 sha256=44be4caa20eb6310a69e9fe9184489eaffab7a7cd433fdd34d1b32615556d0bf
  Stored in directory: C:\Users\david\AppData\Local\Temp\pip-ephem-wheel-cache-g1yc7w4y\wheels\8a\b7\1b\df1e8596f08ef30317

In [2]:
from src_live import Molecule, BasisSet, ELEMENT_SYMBOL, Atom, Shell

In [3]:
benzene_xyz = """12
Benzene molecule
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000
"""
benzene = Molecule.from_string(benzene_xyz)
print(benzene)
print(benzene.to_string())

C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000
12
Generated by Molecule Class
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000


In [4]:
benzene = Molecule.from_string(benzene_xyz)
sto3g = BasisSet("sto-3g")
sto3g.download_from_bse(["C", "H"])
print(sto3g) 

Dump data to sto-3g.json
Basis set: sto-3g
Element: H
l = 0, exponents = [3.42525091 0.62391373 0.1688554 ], coefficients = [0.15432897 0.53532814 0.44463454]
norm_factors = [[1.79444183]
 [0.50032649]
 [0.18773546]]Element: C
l = 0, exponents = [71.61683735 13.04509632  3.53051216], coefficients = [0.15432897 0.53532814 0.44463454]
norm_factors = [[17.54573016]
 [ 4.89210263]
 [ 1.83564365]]l = 0, exponents = [2.94124936 0.6834831  0.22228992], coefficients = [-0.09996723  0.39951283  0.70011547]
norm_factors = [[1.60069643]
 [0.53574229]
 [0.23072779]]l = 1, exponents = [2.94124936 0.6834831  0.22228992], coefficients = [0.15591627 0.60768372 0.39195739]
norm_factors = [[5.49041148 5.49041148 5.49041148]
 [0.88582883 0.88582883 0.88582883]
 [0.21756538 0.21756538 0.21756538]]


In [14]:
from dataclasses import dataclass, field
import copy
import numpy as np

ANG2BOHR = 1.8897259886

@njit(fastmath= True)
def overlap_shell_pair(exp_a: np.ndarray, coeff_a: np.ndarray, norm_a: np.ndarray, center_a: np.ndarray,
                       exp_b: np.ndarray, coeff_b: np.ndarray, norm_b: np.ndarray, center_b: np.ndarray, ijklmn):
    dim_a = norm_a.shape[1]
    dim_b = norm_b.shape[1]
    Dx = center_a[0] - center_b[0]
    Dy = center_a[1] - center_b[1]
    Dz = center_a[2] - center_b[2]
    RAB2 = Dx**2 + Dy**2 + Dz**2
    out = np.zeros((dim_a, dim_b), dtype = np.float64)

    for idx in range(ijklmn.shape[0]):
        i,j,k,l,m,n = ijklmn[idx]
        val = 0
        for pa in range(exp_a,shape[0]):
            c_a = coeff_a[pa] * norm_a[pa, ia]
            alpha_a = exp_a[pa]
            for pb in range(exp_b.shape[0]):
                c_b = coeff_b[pb] * norm_b[pb, ib]
                alpha_b = exp_b[pb]
                P = alpha_a + alpha_b
                Q = alpha_a * alpha_b
                val += ca*cb*S(i,j,k,l,m,n,Dx,Dy,Dz,RAB2,P,Q,alpha_a,alpha_b)




@dataclass
class MolecularIntegrals:
    molecule: Molecule
    basis_set: BasisSet
    shells: list[Shell] = field(init = False)

    def __post_init__(self):
        self.shells = []
        for atom in self.molecule.atoms:
            atom_symbol = ELEMENT_SYMBOL[atom.atomic_number]
            for template_shell in self.basis_set.elements[atom_symbol]:
                shell = copy.deepcopy(template_shell)
                shell.set_center(np.asarray(atom.coord * ANG2BOHR, dtype = np.float64))
                self.shells.append(shell)
            


NameError: name 'njit' is not defined

In [13]:
molints = MolecularIntegrals(benzene, sto3g)
for shell in molints.shells:
    print(shell.center)
len(molints.shells)

[0.         2.65075644 0.        ]
[0.         2.65075644 0.        ]
[0.         2.65075644 0.        ]
[2.29562023 1.32537822 0.        ]
[2.29562023 1.32537822 0.        ]
[2.29562023 1.32537822 0.        ]
[ 2.29562023 -1.32537822  0.        ]
[ 2.29562023 -1.32537822  0.        ]
[ 2.29562023 -1.32537822  0.        ]
[ 0.         -2.65075644  0.        ]
[ 0.         -2.65075644  0.        ]
[ 0.         -2.65075644  0.        ]
[-2.29562023 -1.32537822  0.        ]
[-2.29562023 -1.32537822  0.        ]
[-2.29562023 -1.32537822  0.        ]
[-2.29562023  1.32537822  0.        ]
[-2.29562023  1.32537822  0.        ]
[-2.29562023  1.32537822  0.        ]
[0.         4.70596573 0.        ]
[4.07549645 2.35299231 0.        ]
[ 4.07549645 -2.35299231  0.        ]
[ 0.         -4.70596573  0.        ]
[-4.07549645 -2.35299231  0.        ]
[-4.07549645  2.35299231  0.        ]


24